# OnomaCap Factor Conditioning — canonical Jamo + BF16

HTSAT→BART 구조를 유지하면서 6개 acoustic factor를 factor×time token으로 decoder cross-attention에 추가하고, 한국어 의성어를 `(초성 중성 [종성]?) *` 형식의 canonical Jamo 토큰으로 학습합니다. 이 노트북은 factor 추출, smoke test, 학습, 평가, teacher-forced 자모×factor attention 집계를 순서대로 실행합니다.

In [ ]:
# 의존성 설치
%pip install -q transformers==4.36.2 tokenizers==0.15.2 "huggingface-hub<1.0" torchlibrosa==0.1.0 librosa==0.10.2.post1 ruamel.yaml==0.17.40 gdown==5.2.0 "kagglehub>=0.3.12" pycocoevalcap==1.2 "scikit-learn>=1.4,<1.7" "pandas>=2.1,<2.4" matplotlib seaborn tqdm loguru warmup-scheduler gensim

# 저장소 clone 및 WavCaps overlay 적용
import shutil
import subprocess
from pathlib import Path

ONOMAHOW_REPO = "https://github.com/youhan200203/OnomaHoW.git"
ONOMAHOW_REF = "working"
WAVCAPS_REPO = "https://github.com/XinhaoMei/WavCaps.git"
WAVCAPS_COMMIT = "a5a9649ce305d7fe82cfcf5d6a4a12f03df9ef1e"
ANNOTATION_REPO = "https://github.com/jspirit01/sound-to-onomatopoeia.git"

ONOMAHOW_DIR = Path("/content/OnomaHoW")
WAVCAPS_DIR = Path("/content/WavCaps")
ANNOTATION_DIR = Path("/content/sound-to-onomatopoeia")

for transient_dir in (ONOMAHOW_DIR, WAVCAPS_DIR, ANNOTATION_DIR):
    if transient_dir.exists():
        shutil.rmtree(transient_dir)

subprocess.run(
    ["git", "clone", "-q", "--depth", "1", "--branch", ONOMAHOW_REF, ONOMAHOW_REPO, str(ONOMAHOW_DIR)],
    check=True,
)
subprocess.run(["git", "clone", "-q", WAVCAPS_REPO, str(WAVCAPS_DIR)], check=True)
subprocess.run(
    ["git", "-C", str(WAVCAPS_DIR), "checkout", "--detach", "-q", WAVCAPS_COMMIT],
    check=True,
)
subprocess.run(["git", "clone", "-q", "--depth", "1", ANNOTATION_REPO, str(ANNOTATION_DIR)], check=True)

overlay_root = ONOMAHOW_DIR / "wavcaps_patch"
overlay_files = sorted(overlay_root.rglob("*.py"))
assert overlay_files, "wavcaps_patch overlay가 비어 있습니다."
for source in overlay_files:
    destination = WAVCAPS_DIR / source.relative_to(overlay_root)
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(source, destination)
    assert source.read_bytes() == destination.read_bytes()

helper_destination = WAVCAPS_DIR / "captioning/tools/jamo_preprocessing.py"
config_destination = WAVCAPS_DIR / "captioning/settings/onomacap_jamo.yaml"
factor_config_destination = WAVCAPS_DIR / "captioning/settings/onomacap_jamo_factor.yaml"
shutil.copy2(ONOMAHOW_DIR / "jamo_preprocessing.py", helper_destination)
shutil.copy2(ONOMAHOW_DIR / "configs/onomacap_jamo.yaml", config_destination)
shutil.copy2(ONOMAHOW_DIR / "configs/onomacap_jamo_factor.yaml", factor_config_destination)

checked_out_commit = subprocess.check_output(
    ["git", "-C", str(WAVCAPS_DIR), "rev-parse", "HEAD"], text=True
).strip()
assert checked_out_commit == WAVCAPS_COMMIT
print(f"WavCaps {checked_out_commit} + {len(overlay_files)} overlay files")

In [ ]:
# 공통 설정
import csv
import gc
import json
import os
import random
import sys
import unicodedata

import kagglehub
import numpy as np
import pandas as pd
import torch
from google.colab import drive
from sklearn.model_selection import train_test_split

if str(ONOMAHOW_DIR) not in sys.path:
    sys.path.insert(0, str(ONOMAHOW_DIR))

from jamo_preprocessing import (
    CHOSEONG,
    EXPECTED_LATIN_AUDIO,
    EXPECTED_OUTPUT_ROWS,
    JAMO_VOCAB,
    JONGSEONG,
    JUNGSEONG,
    jamo_to_hangul_caption,
    prepare_rows,
)

SEED = 20

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)
drive.mount("/content/drive")

In [ ]:
# 한국어 annotation 정리 및 canonical Jamo target 생성
audio_root = Path(kagglehub.dataset_download("buraktaci/firat-esc50"))
csv_path = ANNOTATION_DIR / "sound-to-onomatopoeia_annotation.csv"

with csv_path.open(encoding="utf-8-sig", newline="") as stream:
    prepared_rows, processing_report = prepare_rows(csv.DictReader(stream))

assert processing_report.output_rows == EXPECTED_OUTPUT_ROWS == 7_961
assert processing_report.latin_rows_excluded == (EXPECTED_LATIN_AUDIO,)
assert set(processing_report.duplicate_rows_excluded) == {"38_68.wav", "38_95.wav"}
df = pd.DataFrame(prepared_rows)

def audio_key(name):
    return unicodedata.normalize("NFKC", Path(str(name)).name).strip().casefold()

extensions = {".mp3", ".wav", ".flac", ".ogg", ".m4a"}
audio_files = {
    audio_key(path.name): path
    for path in audio_root.rglob("*")
    if path.suffix.lower() in extensions
}
df["audio_path"] = df["audio_file"].map(
    lambda name: str(audio_files.get(audio_key(name), ""))
)

jamo_columns = [f"candidate{index}_jamo" for index in range(1, 6)]
assert len(df) == 7_961
assert df["class"].nunique() == 41
assert not df["audio_file"].duplicated().any()
assert not df[jamo_columns].isna().any().any()
assert not df["audio_path"].eq("").any()
assert df[jamo_columns].map(lambda value: set(value.split()) <= set(JAMO_VOCAB)).all().all()

print(processing_report)
print(df.loc[:, ["audio_file", *jamo_columns]].head(2).to_string(index=False))

In [ ]:
# 6×T acoustic factor 추출: Drive에 정상 파일이 있으면 재사용
FACTOR_OUTPUT = Path("/content/drive/MyDrive/OnomaCap/features/onomacap_acoustic_factors.npz")
LOCAL_FACTOR_OUTPUT = Path("/content/onomacap_acoustic_factors.npz")

def validate_factor_archive(path):
    with np.load(path, allow_pickle=False) as archive:
        expected_files = df["audio_file"].astype(str).to_numpy()
        np.testing.assert_array_equal(archive["audio_files"], expected_files)
        lengths = archive["lengths"]
        offsets = archive["offsets"]
        factor_values = archive["factor_values"]
        assert factor_values.shape[0] == 6
        assert len(lengths) == len(expected_files)
        assert len(offsets) == len(expected_files) + 1
        assert np.array_equal(np.diff(offsets), lengths)
        assert offsets[0] == 0 and offsets[-1] == factor_values.shape[1]
        assert archive["summaries"].shape[0] == len(expected_files)
        return factor_values.shape, lengths.min(), lengths.max()

if FACTOR_OUTPUT.is_file():
    factor_info = validate_factor_archive(FACTOR_OUTPUT)
    print("Existing acoustic factors:", FACTOR_OUTPUT, factor_info)
else:
    extractor = ONOMAHOW_DIR / "extract_acoustic_factors.py"
    assert extractor.is_file(), f"Missing extractor: {extractor}"
    factor_audio_csv = Path("/content/onomacap_factor_audio_files.csv")
    df[["audio_file"]].to_csv(factor_audio_csv, index=False)
    subprocess.run(
        [
            sys.executable, str(extractor),
            "--annotation-csv", str(factor_audio_csv),
            "--audio-root", str(audio_root),
            "--output", str(LOCAL_FACTOR_OUTPUT),
            "--drive-output", str(FACTOR_OUTPUT),
            "--overwrite",
        ],
        check=True,
    )
    factor_info = validate_factor_archive(FACTOR_OUTPUT)
    print("Extracted acoustic factors:", FACTOR_OUTPUT, factor_info)

In [ ]:
# class-stratified 80/10/10 split
train_df, rest_df = train_test_split(
    df, test_size=0.2, stratify=df["class"], random_state=SEED
)
val_df, test_df = train_test_split(
    rest_df, test_size=0.5, stratify=rest_df["class"], random_state=SEED
)
splits = {
    "train": train_df.reset_index(drop=True),
    "val": val_df.reset_index(drop=True),
    "test": test_df.reset_index(drop=True),
}
assert {name: len(frame) for name, frame in splits.items()} == {
    "train": 6_368, "val": 796, "test": 797
}
print({name: len(frame) for name, frame in splits.items()})

In [ ]:
# WavCaps JSON 작성: 각 caption은 공백으로 구분된 canonical Jamo
JSON_DIR = WAVCAPS_DIR / "captioning/data/OnomaCap/json_files"
JSON_DIR.mkdir(parents=True, exist_ok=True)

def to_wavcaps_item(row):
    return {
        "audio": str(Path(row["audio_path"]).resolve()),
        **{
            f"caption_{index}": str(row[f"candidate{index}_jamo"])
            for index in range(1, 6)
        },
    }

maximum_target_tokens = 0
for split_name, frame in splits.items():
    records = [to_wavcaps_item(row) for _, row in frame.iterrows()]
    output_path = JSON_DIR / f"{split_name}.json"
    output_path.write_text(
        json.dumps({"data": records}, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    maximum_target_tokens = max(
        maximum_target_tokens,
        max(len(record[f"caption_{index}"].split()) for record in records for index in range(1, 6)),
    )
    print(output_path, len(records))

assert maximum_target_tokens + 2 <= 80  # BOS/EOS 포함
assert len(splits["train"]) * 5 == 31_840
print("maximum Jamo target tokens (without BOS/EOS):", maximum_target_tokens)

In [ ]:
# HTSAT 사전학습 checkpoint 배치
HTSAT_SOURCE = Path("/content/drive/MyDrive/OnomaCap/pretrained/HTSAT.ckpt")
HTSAT_DESTINATION = WAVCAPS_DIR / "captioning/pretrained_models/audio_encoder/HTSAT.ckpt"
if not HTSAT_SOURCE.is_file():
    raise FileNotFoundError(
        f"{HTSAT_SOURCE}가 없습니다. Drive에 HTSAT.ckpt를 먼저 올려주세요."
    )
HTSAT_DESTINATION.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(HTSAT_SOURCE, HTSAT_DESTINATION)
assert HTSAT_DESTINATION.stat().st_size == HTSAT_SOURCE.stat().st_size
print(HTSAT_DESTINATION)

In [ ]:
# 데이터/자모 문법 기본 검증
sample_jamo = "ᄐ ᅡ ᄃ ᅳ ᄅ ᅳ ᆼ"
assert jamo_to_hangul_caption(sample_jamo) == "타 드 릉"
assert "ᄀ" != "ᆨ"  # 같은 ㄱ 계열이어도 초성과 종성은 별도 codepoint/token

sample_record = json.loads((JSON_DIR / "train.json").read_text(encoding="utf-8"))["data"][0]
for index in range(1, 6):
    assert jamo_to_hangul_caption(sample_record[f"caption_{index}"])
print(sample_record["caption_1"], "->", jamo_to_hangul_caption(sample_record["caption_1"]))

In [ ]:
# 본 학습 전 실제 1배치 BF16 forward/backward + constrained generation smoke test
import ruamel.yaml as yaml
from torch.utils.data import DataLoader, Subset

SMOKE_MARKER = Path("/content/onomacap_jamo_bf16_smoke_passed")
SMOKE_MARKER.unlink(missing_ok=True)
previous_cwd = Path.cwd()
smoke_model = None

try:
    captioning_dir = WAVCAPS_DIR / "captioning"
    os.chdir(captioning_dir)
    if str(captioning_dir) not in sys.path:
        sys.path.insert(0, str(captioning_dir))

    from data_handling.datamodule import AudioCaptionDataModule, collate_fn
    from models.bart_captioning import BartCaptionModel
    from pretrain import validate
    from tools.optim_utils import get_optimizer
    from tools.utils import setup_seed

    with open("settings/onomacap_jamo_factor.yaml", "r") as stream:
        smoke_config = yaml.safe_load(stream)
    smoke_config["data_args"] = dict(smoke_config["data_args"])
    smoke_config["data_args"]["batch_size"] = 2
    smoke_config["data_args"]["num_workers"] = 0
    setup_seed(smoke_config["seed"])

    assert torch.cuda.is_available(), "CUDA GPU가 필요합니다."
    assert torch.cuda.is_bf16_supported(), "BF16 지원 CUDA GPU가 필요합니다."
    smoke_device = "cuda"

    smoke_data = AudioCaptionDataModule(smoke_config, "OnomaCap")
    audio, text, _, _, factors, factor_mask = next(iter(smoke_data.train_dataloader()))
    smoke_model = BartCaptionModel(smoke_config).to(smoke_device)
    assert factors.shape[:2] == (2, 6)
    assert factor_mask.shape == (2, factors.shape[-1])
    assert len(smoke_model.jamo_to_id) == 67
    assert smoke_model.jamo_to_id["ᄀ"] != smoke_model.jamo_to_id["ᆨ"]
    assert all(
        smoke_model.tokenizer.convert_tokens_to_ids(token) == token_id
        for token, token_id in smoke_model.jamo_to_id.items()
    )

    optimizer = get_optimizer(
        smoke_model.parameters(),
        lr=smoke_config["optim_args"]["lr"],
        betas=smoke_config["optim_args"]["betas"],
        eps=smoke_config["optim_args"]["eps"],
        momentum=smoke_config["optim_args"]["momentum"],
        weight_decay=smoke_config["optim_args"]["weight_decay"],
        optimizer_name=smoke_config["optim_args"]["optimizer_name"],
    )
    optimizer.zero_grad(set_to_none=True)
    with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
        smoke_loss = smoke_model(
            audio.to(smoke_device),
            text,
            factors=factors.to(smoke_device),
            factor_mask=factor_mask.to(smoke_device),
        )
    assert torch.isfinite(smoke_loss)
    smoke_loss.backward()
    gradient_norm = torch.nn.utils.clip_grad_norm_(
        smoke_model.parameters(), 2.0, error_if_nonfinite=True
    )
    optimizer.step()
    with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
        smoke_attention = smoke_model.teacher_forced_factor_attention(
            audio[:1].to(smoke_device),
            text[:1],
            factors[:1].to(smoke_device),
            factor_mask[:1].to(smoke_device),
        )
    smoke_factor_mass = smoke_attention["factor_attention_mass"].float()
    smoke_total_mass = smoke_attention["total_factor_attention"].float()
    assert smoke_factor_mass.shape[-1] == 6
    assert torch.isfinite(smoke_factor_mass).all()
    assert torch.isfinite(smoke_total_mass).all()
    assert smoke_total_mass.min() >= 0.0
    assert smoke_total_mass.max() <= 1.0001

    smoke_val_loader = DataLoader(
        Subset(smoke_data.val_set, range(2)),
        batch_size=2,
        shuffle=False,
        num_workers=0,
        collate_fn=collate_fn,
    )
    smoke_log_dir = Path("outputs/smoke")
    smoke_log_dir.mkdir(parents=True, exist_ok=True)
    smoke_metrics = validate(
        smoke_val_loader, smoke_model, smoke_device, smoke_log_dir, 0, 1
    )
    assert set(smoke_metrics) == {"bleu_1", "bleu_2", "bleu_3", "bleu_4", "meteor", "rouge_l"}
    SMOKE_MARKER.write_text("passed", encoding="utf-8")
    print("BF16 smoke loss:", float(smoke_loss.detach().cpu()))
    print("gradient norm:", float(gradient_norm.detach().cpu()))
finally:
    os.chdir(previous_cwd)
    del smoke_model
    gc.collect()
    torch.cuda.empty_cache()

assert SMOKE_MARKER.is_file()

In [ ]:
# 학습: epoch checkpoint가 있으면 모델/optimizer/RNG를 복원해 자동 resume
assert Path("/content/onomacap_jamo_bf16_smoke_passed").is_file()
captioning_dir = WAVCAPS_DIR / "captioning"
training_environment = os.environ.copy()
training_environment["PYTHONPATH"] = str(captioning_dir)
subprocess.run(
    [
        sys.executable,
        "train.py",
        "--exp_name", "onomacap_htsat_bart_jamo_factor_bf16",
        "--config", "settings/onomacap_jamo_factor.yaml",
        "--lr", "3e-5",
        "--seed", "20",
    ],
    cwd=captioning_dir,
    env=training_environment,
    check=True,
)

In [ ]:
# best validation checkpoint 확인
FOLDER_NAME = "onomacap_htsat_bart_jamo_factor_bf16_lr_3e-05_batch_16_seed_20"
CHECKPOINT_DIR = Path("/content/drive/MyDrive/OnomaCap/checkpoints") / FOLDER_NAME
BEST_PATH = CHECKPOINT_DIR / "best_model.pt"
best = torch.load(BEST_PATH, map_location="cpu", weights_only=False)
print("best epoch:", best["epoch"])
print("selection metric:", best["selection_metric"])
print("best val BLEU-1:", best["selection_score"])
print("all val scores:", best["val_scores"])

In [ ]:
# 원하는 epoch checkpoint를 test set에서 평가
TEST_EPOCH = 10
TEST_BEAM_SIZE = 1

test_previous_cwd = Path.cwd()
test_model = None
try:
    captioning_dir = WAVCAPS_DIR / "captioning"
    os.chdir(captioning_dir)
    if str(captioning_dir) not in sys.path:
        sys.path.insert(0, str(captioning_dir))

    from data_handling.datamodule import AudioCaptionDataModule
    from models.bart_captioning import BartCaptionModel
    from pretrain import validate
    from loguru import logger
    logger.remove()
    logger.add(sys.stdout, filter=lambda record: record["extra"].get("indent") == 1)

    with open("settings/onomacap_jamo_factor.yaml", "r") as stream:
        test_config = yaml.safe_load(stream)
    test_data = AudioCaptionDataModule(test_config, "OnomaCap")
    test_model = BartCaptionModel(test_config).to("cuda")

    checkpoint_path = CHECKPOINT_DIR / "epochs" / f"epoch_{TEST_EPOCH:02d}.pt"
    checkpoint = torch.load(checkpoint_path, map_location="cuda", weights_only=False)
    test_model.load_state_dict(checkpoint["model"])

    test_log_dir = Path(test_config["results"]["root"]) / FOLDER_NAME / f"epoch_{TEST_EPOCH:02d}"
    test_log_dir.mkdir(parents=True, exist_ok=True)
    test_metrics = validate(
        test_data.test_dataloader(),
        test_model,
        device="cuda",
        log_dir=test_log_dir,
        epoch=TEST_EPOCH,
        beam_size=TEST_BEAM_SIZE,
    )
    print({name: values["score"] for name, values in test_metrics.items()})
finally:
    os.chdir(test_previous_cwd)
    del test_model
    gc.collect()
    torch.cuda.empty_cache()

In [ ]:
# test set teacher forcing: 모든 자모 occurrence의 factor attention 집계
import matplotlib.pyplot as plt
import seaborn as sns

attention_previous_cwd = Path.cwd()
attention_model = None
try:
    captioning_dir = WAVCAPS_DIR / "captioning"
    os.chdir(captioning_dir)
    if str(captioning_dir) not in sys.path:
        sys.path.insert(0, str(captioning_dir))

    from data_handling.datamodule import AudioCaptionDataModule
    from models.bart_captioning import BartCaptionModel

    with open("settings/onomacap_jamo_factor.yaml", "r") as stream:
        attention_config = yaml.safe_load(stream)
    attention_data = AudioCaptionDataModule(attention_config, "OnomaCap")
    attention_model = BartCaptionModel(attention_config).to("cuda")
    checkpoint = torch.load(BEST_PATH, map_location="cpu", weights_only=False)
    attention_model.load_state_dict(checkpoint["model"])
    attention_model.eval()

    token_order = list(JAMO_VOCAB)
    token_to_row = {token: index for index, token in enumerate(token_order)}
    factor_names = list(attention_model.factor_names)
    factor_mass_sum = np.zeros((len(token_order), len(factor_names)), dtype=np.float64)
    total_factor_mass_sum = np.zeros(len(token_order), dtype=np.float64)
    occurrence_count = np.zeros(len(token_order), dtype=np.int64)

    for batch in attention_data.test_dataloader():
        audios, caption_lists, _, _, factors, factor_mask = batch
        audios = audios.to("cuda", non_blocking=True)
        factors = factors.to("cuda", non_blocking=True)
        factor_mask = factor_mask.to("cuda", non_blocking=True)

        for reference_index in range(5):
            captions = [items[reference_index] for items in caption_lists]
            with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
                attention = attention_model.teacher_forced_factor_attention(
                    audios, captions, factors, factor_mask, layer_index=-1
                )
            target_ids = attention["target_ids"].detach().cpu()
            target_mask = attention["target_mask"].detach().cpu()
            factor_mass = attention["factor_attention_mass"].float().detach().cpu()
            total_mass = attention["total_factor_attention"].float().detach().cpu()

            for token, token_id in attention_model.jamo_to_id.items():
                selected = target_mask & target_ids.eq(token_id)
                count = int(selected.sum())
                if count == 0:
                    continue
                row = token_to_row[token]
                factor_mass_sum[row] += factor_mass[selected].sum(dim=0).numpy()
                total_factor_mass_sum[row] += float(total_mass[selected].sum())
                occurrence_count[row] += count

    mean_factor_mass = np.divide(
        factor_mass_sum, occurrence_count[:, None],
        out=np.zeros_like(factor_mass_sum), where=occurrence_count[:, None] > 0,
    )
    factor_share = np.divide(
        mean_factor_mass, mean_factor_mass.sum(axis=1, keepdims=True),
        out=np.zeros_like(mean_factor_mass),
        where=mean_factor_mass.sum(axis=1, keepdims=True) > 0,
    )
    mean_total_factor_mass = np.divide(
        total_factor_mass_sum, occurrence_count,
        out=np.zeros_like(total_factor_mass_sum), where=occurrence_count > 0,
    )

    category_by_token = {
        **{token: "choseong" for token in CHOSEONG},
        **{token: "jungseong" for token in JUNGSEONG},
        **{token: "jongseong" for token in JONGSEONG},
    }
    rows = []
    for index, token in enumerate(token_order):
        row = {
            "jamo": token,
            "category": category_by_token[token],
            "occurrences": int(occurrence_count[index]),
            "mean_total_factor_attention": mean_total_factor_mass[index],
        }
        row.update({f"{name}_mass": mean_factor_mass[index, f] for f, name in enumerate(factor_names)})
        row.update({f"{name}_share": factor_share[index, f] for f, name in enumerate(factor_names)})
        rows.append(row)
    attention_table = pd.DataFrame(rows)

    attention_output_dir = Path(attention_config["results"]["root"]) / FOLDER_NAME / "factor_attention"
    attention_output_dir.mkdir(parents=True, exist_ok=True)
    attention_table.to_csv(attention_output_dir / "teacher_forced_jamo_factor_attention.csv", index=False)
    display(attention_table)

    groups = [("choseong", CHOSEONG), ("jungseong", JUNGSEONG), ("jongseong", JONGSEONG)]
    for category, tokens in groups:
        indices = [token_to_row[token] for token in tokens if occurrence_count[token_to_row[token]] > 0]
        labels = [
            f"{token_order[index]}  n={occurrence_count[index]}  factor-mass={mean_total_factor_mass[index]:.3f}"
            for index in indices
        ]
        figure, axis = plt.subplots(figsize=(10, max(5, len(indices) * 0.42)))
        sns.heatmap(
            factor_share[indices], annot=True, fmt=".2f", cmap="mako",
            vmin=0.0, vmax=1.0, xticklabels=factor_names, yticklabels=labels, ax=axis,
        )
        axis.set_title(
            f"Teacher-forced {category} factor attention share\n"
            "last decoder layer, mean over heads, every token occurrence counted"
        )
        axis.set_xlabel("Acoustic factor")
        axis.set_ylabel("Canonical Jamo")
        figure.tight_layout()
        figure.savefig(attention_output_dir / f"{category}_factor_attention.png", dpi=200, bbox_inches="tight")
        plt.show()
finally:
    os.chdir(attention_previous_cwd)
    if attention_model is not None:
        del attention_model
    gc.collect()
    torch.cuda.empty_cache()